import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson
import scipy.stats as stats

sns.set_theme(style="whitegrid")
PROC = "../data/processed"

np.random.seed(42)

oil = pd.read_csv(f"{PROC}/oil_panel.csv", parse_dates=["date"], index_col="date")
wheatp = pd.read_csv(f"{PROC}/wheat_panel.csv", parse_dates=["date"], index_col="date")
copperp = pd.read_csv(f"{PROC}/copper_panel.csv", parse_dates=["date"], index_col="date")

markets = {
    "Oil": oil["wti_price_real"],
    "Wheat": wheatp["wheat_price_real"],
    "Copper": copperp["copper_price_real"]
}

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson
import scipy.stats as stats

sns.set_theme(style="whitegrid", context="notebook")
PROC = "../data/processed"
np.random.seed(42)

oil     = pd.read_csv(f"{PROC}/oil_panel.csv",    parse_dates=["date"], index_col="date")
wheatp  = pd.read_csv(f"{PROC}/wheat_panel.csv",  parse_dates=["date"], index_col="date")
copperp = pd.read_csv(f"{PROC}/copper_panel.csv", parse_dates=["date"], index_col="date")

markets = {
    "Oil":    oil["wti_price_real"],
    "Wheat":  wheatp["wheat_price_real"],
    "Copper": copperp["copper_price_real"],
}



## 4. Market shock analysis

Here I study selected large shocks in oil, wheat, and copper and measure how prices move relative to a pre-shock baseline.


In [ ]:
def shock_stats(series, shock_start, pre_months=6, window_months=24, recovery_band=0.10):
    series = series.sort_index().dropna()
    shock_start = pd.Timestamp(shock_start)

    pre = series[
        (series.index < shock_start) &
        (series.index >= shock_start - pd.DateOffset(months=pre_months))
    ]
    baseline = pre.mean()

    post = series[
        (series.index >= shock_start) &
        (series.index < shock_start + pd.DateOffset(months=window_months))
    ]

    if post.empty or pd.isna(baseline):
        return None

    deviation = (post - baseline) / baseline
    peak = deviation.abs().idxmax()
    peak_deviation = deviation.loc[peak]

    months_to_peak = (peak.to_period("M") - shock_start.to_period("M")).n
    after_peak = post[post.index >= peak]
    recovered = after_peak[(after_peak - baseline).abs() / baseline <= recovery_band]

    months_to_recover = np.nan
    if not recovered.empty:
        months_to_recover = (
            recovered.index[0].to_period("M") - shock_start.to_period("M")
        ).n

    return {
        "baseline": baseline,
        "peak_dev_pct": peak_deviation * 100,
        "months_to_peak": months_to_peak,
        "months_to_recover": months_to_recover,
        "post": post,
        "dev": deviation * 100
    }

SHOCKS = {
    "Oil": {
        "2008 Global Financial Crisis": "2008-07-01",
        "2014-16 Oil Price Crash": "2014-06-01",
        "2020 COVID-19 Demand Shock": "2020-02-01",
        "2022 Russia-Ukraine War": "2022-02-01"
    },
    "Wheat": {
        "2007-08 Global Food Price Crisis": "2007-06-01",
        "2010-11 Russia Export Ban / Arab Spring": "2010-07-01",
        "2020 COVID-19 Shock": "2020-02-01",
        "2022 Russia-Ukraine War": "2022-02-01"
    },
    "Copper": {
        "2008 Global Financial Crisis": "2008-07-01",
        "2020 COVID-19 Shock": "2020-02-01",
        "2021-22 Post-COVID Demand Surge": "2021-01-01"
    }
}

shock_rows = []
shock_paths = {}

for market, events in SHOCKS.items():
    for event, start in events.items():
        result = shock_stats(markets[market], start)
        if result is None:
            continue

        shock_paths[(market, event)] = result
        shock_rows.append({
            "Market": market,
            "Event": event,
            "Start": start,
            "Peak deviation (%)": round(result["peak_dev_pct"], 1),
            "Months to peak": result["months_to_peak"],
            "Months to recover (10% band)": result["months_to_recover"]
        })

shock_table = pd.DataFrame(shock_rows)
shock_table


For each selected event, I record the largest deviation and whether the price returns within 10% of the baseline during the 24-month window.



In [ ]:

fig, axes = plt.subplots(4, 3, figsize=(14, 13))
axes = axes.flatten()
i = 0
for (market, event), res in shock_paths.items():
    ax = axes[i]
    ax.plot(res["dev"].index, res["dev"].values, color="#1f77b4")
    ax.axhline(0, color="k", lw=1, ls="--")
    ax.axhline(10, color="gray", lw=0.7, ls=":")
    ax.axhline(-10, color="gray", lw=0.7, ls=":")
    ax.set_title(f"{market}: {event}", fontsize=9)
    ax.set_ylabel("% dev. from baseline")
    i += 1
for j in range(i, len(axes)):
    axes[j].axis("off")
plt.tight_layout()
plt.show()



## 5. Mean reversion

I also test whether prices tend to move back toward their recent trend.


In [ ]:
def mean_reversion_fit(price_real, trend_window=36):
    log_price = np.log(price_real.dropna())
    trend = log_price.rolling(
        trend_window,
        center=True,
        min_periods=trend_window // 2
    ).mean()

    deviation = (log_price - trend).dropna()
    lagged = deviation.shift(1)
    change = deviation - lagged

    df = pd.concat([lagged, change], axis=1).dropna()
    df.columns = ["lag", "delta"]

    X = sm.add_constant(df["lag"])
    model = sm.OLS(df["delta"], X).fit(
        cov_type="HAC", cov_kwds={"maxlags": 6}
    )

    beta = model.params["lag"]
    half_life = np.nan
    if -1 < beta < 0:
        half_life = np.log(0.5) / np.log(1 + beta)

    return {
        "model": model,
        "beta": beta,
        "p_value": model.pvalues["lag"],
        "half_life_months": half_life,
        "dev": deviation,
        "n": len(df)
    }

mr_results = {name: mean_reversion_fit(series) for name, series in markets.items()}

mr_table = pd.DataFrame({
    name: {
        "AR(1) coefficient": round(result["beta"], 4),
        "p-value": round(result["p_value"], 4),
        "Half-life (months)": round(result["half_life_months"], 1),
        "N": result["n"]
    }
    for name, result in mr_results.items()
}).T

mr_table


A negative coefficient is consistent with movement back toward the trend. The half-life gives an approximate measure of how quickly this happens.


### 5.1 Bootstrap confidence intervals

I use a block bootstrap to get uncertainty intervals for the mean-reversion estimates.


In [ ]:
def block_bootstrap_halflife(dev, n_boot=2000, block=12, seed=42):
    rng = np.random.RandomState(seed)
    values = dev.dropna().values
    n = len(values)
    betas = []

    for _ in range(n_boot):
        indices = []
        while len(indices) < n:
            start = rng.randint(0, max(1, n - block))
            indices.extend(range(start, min(start + block, n)))

        sample = pd.Series(values[indices[:n]])
        lagged = sample.shift(1)
        change = sample - lagged
        df = pd.concat([lagged, change], axis=1).dropna()
        df.columns = ["lag", "delta"]

        if len(df) < 10:
            continue

        X = sm.add_constant(df["lag"])
        betas.append(sm.OLS(df["delta"], X).fit().params["lag"])

    betas = np.array(betas)
    beta_ci = np.percentile(betas, [2.5, 50, 97.5])

    half_lives = np.where(
        (betas > -1) & (betas < 0),
        np.log(0.5) / np.log(1 + betas),
        np.nan
    )
    hl_ci = np.nanpercentile(half_lives, [2.5, 50, 97.5])

    return beta_ci, hl_ci, betas

boot_rows = []
for name, result in mr_results.items():
    beta_ci, hl_ci, betas = block_bootstrap_halflife(result["dev"])
    boot_rows.append({
        "Market": name,
        "beta (median)": round(beta_ci[1], 4),
        "beta 95% CI": f"[{beta_ci[0]:.4f}, {beta_ci[2]:.4f}]",
        "Half-life (median, mo.)": round(hl_ci[1], 1),
        "Half-life 95% CI (mo.)": f"[{hl_ci[0]:.1f}, {hl_ci[2]:.1f}]"
    })

boot_table = pd.DataFrame(boot_rows)
boot_table


The bootstrap gives a second way to assess how stable the estimates are.


### 5.2 Robustness checks

I repeat the mean-reversion calculation using different samples and trend windows.


**(a) Different time periods**

I compare the full sample with earlier and later periods.


In [ ]:
def mean_rev_beta(price_real, start=None, end=None, trend_window=36):
    series = price_real.copy()
    if start:
        series = series[series.index >= start]
    if end:
        series = series[series.index < end]

    if len(series.dropna()) < trend_window + 12:
        return np.nan, np.nan

    result = mean_reversion_fit(series, trend_window)
    return result["beta"], result["half_life_months"]


**(b) Different trend windows**

I check whether the result changes when the trend is calculated over 24, 36, or 48 months.


In [ ]:
window_rows = []

for name, series in markets.items():
    for window in [24, 36, 48]:
        beta, half_life = mean_rev_beta(series, trend_window=window)
        window_rows.append({
            "Market": name,
            "Trend window (months)": window,
            "beta": round(beta, 4),
            "Half-life (months)": round(half_life, 1)
        })

pd.DataFrame(window_rows).pivot(
    index="Market",
    columns="Trend window (months)",
    values="Half-life (months)"
)


**(c) Removing extreme oil shocks**

I repeat the oil growth regression after removing several extreme months.


In [ ]:
d = oil.dropna(
    subset=["wti_price_usd_per_bbl", "oil_production_kbd"]
).copy()
d["log_price"] = np.log(d["wti_price_usd_per_bbl"])
d["log_prod"] = np.log(d["oil_production_kbd"])
d["dlog_price"] = d["log_price"].diff()
d["dlog_prod"] = d["log_prod"].diff()
dd = d.dropna(subset=["dlog_price", "dlog_prod"])

extreme_months = (
    pd.period_range("2008-09", "2009-02", freq="M")
    .union(pd.period_range("2020-03", "2020-04", freq="M"))
)

trimmed = dd[~dd.index.to_period("M").isin(extreme_months)]

def fit_growth(data):
    X = sm.add_constant(data["dlog_prod"])
    return sm.OLS(
        data["dlog_price"], X
    ).fit(cov_type="HAC", cov_kwds={"maxlags": 6})

full_model = fit_growth(dd)
trimmed_model = fit_growth(trimmed)

print(f"Full sample: slope={full_model.params['dlog_prod']:.3f}, p={full_model.pvalues['dlog_prod']:.4g}")
print(f"Extreme months removed: slope={trimmed_model.params['dlog_prod']:.3f}, p={trimmed_model.pvalues['dlog_prod']:.4g}")


The robustness checks show how sensitive the results are to the sample and modelling choices.


### 5.3 Cross-market comparison

I compare the three markets using the estimated half-life and the shock-recovery results.


In [ ]:

cross = pd.DataFrame({
    "Market": ["Oil", "Wheat", "Copper"],
    "Half-life, mean reversion (mo.)": [round(mr_results[m]["half_life_months"], 1) for m in ["Oil","Wheat","Copper"]],
    "Median # shock events recovered <24m": [
        shock_table[shock_table.Market == m]["Months to recover (10% band)"].notna().sum()
        for m in ["Oil", "Wheat", "Copper"]
    ],
    "Total shock events studied": [shock_table[shock_table.Market == m].shape[0] for m in ["Oil","Wheat","Copper"]],
    "Avg. |peak deviation| (%)": [
        round(shock_table[shock_table.Market == m]["Peak deviation (%)"].abs().mean(), 1) for m in ["Oil","Wheat","Copper"]
    ],
})
cross["Structural market feature"] = [
    "Deep financial market, OPEC+ can adjust supply quickly, U.S. shale is a fast swing producer",
    "Seasonal harvest cycles, but export bans / war can remove large volumes for years",
    "Supply concentrated in few countries (Chile, Peru, DRC); new mine capacity takes years to add",
]
cross



In [ ]:

fig, ax = plt.subplots(figsize=(7, 4))
colors = ["#1f77b4", "#2ca02c", "#d62728"]
ax.bar(cross["Market"], cross["Half-life, mean reversion (mo.)"], color=colors)
ax.set_ylabel("Half-life (months)")
ax.set_title("Speed of self-correction by market\n(lower = faster reversion to trend)")
for i, v in enumerate(cross["Half-life, mean reversion (mo.)"]):
    ax.text(i, v + 0.1, str(v), ha="center")
plt.tight_layout()
plt.show()



The markets do not adjust at the same speed. This comparison is descriptive rather than a formal ranking of market efficiency.


The 10% recovery band is a simple rule chosen for this project. Different recovery bands could give somewhat different results.

## 6. Market failure discussion

A price returning toward its previous trend does not automatically mean the market was inefficient. Prices can adjust because supply, demand, expectations, and other conditions change.



## Model evaluation and diagnostics

I check the main models using residual plots and standard diagnostic tests.


In [ ]:

# Re-fit the two headline models for diagnostics
d = oil.dropna(subset=["wti_price_usd_per_bbl", "oil_production_kbd"]).copy()
d["log_price"] = np.log(d["wti_price_usd_per_bbl"]); d["log_prod"] = np.log(d["oil_production_kbd"])
d["dlog_price"] = d["log_price"].diff(); d["dlog_prod"] = d["log_prod"].diff()
dd = d.dropna(subset=["dlog_price", "dlog_prod"])
X = sm.add_constant(dd["dlog_prod"])
h1_model = sm.OLS(dd["dlog_price"], X).fit()

mr_oil = mean_reversion_fit(oil["wti_price_real"])
mr_model = mr_oil["model"]

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for row, (name, model, resid_label) in enumerate([
    ("H1: oil price growth ~ production growth", h1_model, "H1 residuals"),
    ("Mean-reversion: oil deviation AR(1)", mr_model, "AR(1) residuals"),
]):
    resid = model.resid
    fitted = model.fittedvalues
    axes[row, 0].scatter(fitted, resid, alpha=0.4, s=12)
    axes[row, 0].axhline(0, color="k", lw=1)
    axes[row, 0].set_title(f"{name}\nResiduals vs. fitted")
    axes[row, 0].set_xlabel("Fitted"); axes[row, 0].set_ylabel("Residual")

    stats.probplot(resid, dist="norm", plot=axes[row, 1])
    axes[row, 1].set_title("Normal Q-Q plot")

    axes[row, 2].hist(resid, bins=30, color="#1f77b4", alpha=0.8)
    axes[row, 2].set_title("Residual distribution")
plt.tight_layout()
plt.show()



In [ ]:

diag_rows = []
for name, model, X_ in [("H1 (price growth ~ prod growth)", h1_model, X),
                          ("Mean-reversion AR(1), oil", mr_model, sm.add_constant(mr_oil["dev"].shift(1).dropna()))]:
    bp_stat, bp_p, _, _ = het_breuschpagan(model.resid, model.model.exog)
    dw = durbin_watson(model.resid)
    jb_stat, jb_p = stats.jarque_bera(model.resid)[:2]
    diag_rows.append({
        "Model": name,
        "Breusch-Pagan p (H0: homoskedastic)": round(bp_p, 4),
        "Durbin-Watson (2=no autocorr.)": round(dw, 2),
        "Jarque-Bera p (H0: normal resid.)": round(jb_p, 4),
    })
pd.DataFrame(diag_rows)



The diagnostics help identify issues such as changing variance, residual autocorrelation, and non-normality.


## Limitations

This is an observational project, so it does not establish causality. The choice of shocks, baseline period, recovery band, trend window, and sample can affect the results. The three markets also have different economic structures.



## Conclusion

The analysis finds evidence that some large commodity-price movements are followed by adjustment toward earlier levels or trends. The speed and size of adjustment vary across markets. The results should be interpreted as descriptive evidence rather than proof that markets always correct themselves.
